# CareerPath AI — Model Training & Evaluation
**Dataset:** student_career_data.csv | **Algorithm:** Random Forest Classifier

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json, joblib, os
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
print('Libraries loaded successfully.')

## 1. Load & Inspect Dataset

In [ ]:
df = pd.read_csv('student_career_data.csv')
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
print('\nFirst 5 rows:')
df.head()

In [ ]:
print('=== Descriptive Statistics ===')
print(df.describe())
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Career Distribution ===')
print(df['career'].value_counts())

## 2. Exploratory Data Analysis (EDA)

In [ ]:
import os
# 2.1 Career Distribution Bar Chart
plt.figure(figsize=(10, 5))
career_counts = df['career'].value_counts()
sns.barplot(x=career_counts.index, y=career_counts.values, palette='viridis')
plt.title('Career Distribution', fontsize=16)
plt.xlabel('Career Path')
plt.ylabel('Number of Students')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(os.path.join('viz', 'career_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import os
# 2.2 GPA Distribution
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.histplot(df['gpa'], kde=True, bins=30, ax=axes[0,0], color='skyblue')
axes[0,0].set_title('GPA Distribution')

sns.histplot(df['internships'], kde=True, bins=10, ax=axes[0,1], color='lightcoral')
axes[0,1].set_title('Internships Distribution')

sns.histplot(df['projects'], kde=True, bins=15, ax=axes[1,0], color='lightgreen')
axes[1,0].set_title('Projects Distribution')

sns.histplot(df['leadership'], kde=True, bins=5, ax=axes[1,1], color='gold')
axes[1,1].set_title('Leadership Roles Distribution')

plt.suptitle('Numerical Feature Distributions', fontsize=16)
plt.tight_layout()
plt.savefig(os.path.join('viz', 'feature_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import os
# 2.3 GPA by Career (Boxplot)
plt.figure(figsize=(12, 6))
sns.boxplot(x='career', y='gpa', data=df, palette='Set2')
plt.title('GPA Distribution by Career Path', fontsize=16)
plt.xlabel('Career')
plt.ylabel('GPA')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(os.path.join('viz', 'gpa_by_career.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import os
# 2.4 Career Distribution by Major
plt.figure(figsize=(12, 6))
career_major = pd.crosstab(df['major'], df['career'])
career_major.plot(kind='bar', stacked=True, figsize=(12, 6), colormap='viridis')
plt.title('Career Distribution by Major', fontsize=16)
plt.xlabel('Major')
plt.ylabel('Count')
plt.xticks(rotation=30, ha='right')
plt.legend(title='Career', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(os.path.join('viz', 'career_by_major.png'), dpi=150, bbox_inches='tight')
plt.show()

## 3. Feature Engineering

In [ ]:
# 3.1 Encode target label
le_career = LabelEncoder()
y_encoded = le_career.fit_transform(df['career'])
print('Career classes:', le_career.classes_)

# 3.2 One-hot encode skills
skills_df = df['skills'].str.get_dummies(sep=',')
# Strip whitespace from column names that may come from CSV
skills_df.columns = [c.strip() for c in skills_df.columns]
print('Skills encoded:', skills_df.columns.tolist())

# 3.3 Encode major
le_major = LabelEncoder()
major_encoded = le_major.fit_transform(df['major'])
print('Major classes:', le_major.classes_)

# 3.4 Build final feature matrix
X = pd.DataFrame({
    'gpa': df['gpa'],
    'internships': df['internships'],
    'projects': df['projects'],
    'leadership': df['leadership'],
    'major_encoded': major_encoded
})
X = pd.concat([X, skills_df], axis=1)
print('\nFinal feature matrix shape:', X.shape)
print('Feature names:', X.columns.tolist())

In [ ]:
import os
# 3.5 Correlation Heatmap
numeric_cols = ['gpa', 'internships', 'projects', 'leadership', 'major_encoded']
corr_df = X[numeric_cols].copy()
corr_df['career_encoded'] = y_encoded

plt.figure(figsize=(10, 8))
sns.heatmap(corr_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Feature Correlation Heatmap', fontsize=16)
plt.tight_layout()
plt.savefig(os.path.join('viz', 'correlation_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

## 4. Model Training

In [ ]:
# 4.1 Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)
print(f'Training samples: {X_train.shape[0]}')
print(f'Test samples:     {X_test.shape[0]}')

In [ ]:
# 4.2 Train Random Forest
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)
print('Model trained successfully.')

## 5. Evaluation

In [ ]:
# 5.1 Classification Metrics
y_pred = model.predict(X_test)

accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall    = recall_score(y_test, y_pred, average='weighted')
f1        = f1_score(y_test, y_pred, average='weighted')

print(f'Accuracy:  {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall:    {recall:.4f}')
print(f'F1-Score:  {f1:.4f}')
print()
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=le_career.classes_))

In [ ]:
import os
# 5.2 Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le_career.classes_,
            yticklabels=le_career.classes_)
plt.title('Confusion Matrix', fontsize=16)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(os.path.join('viz', 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 5.3 Cross-Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X, y_encoded, cv=cv, scoring='accuracy')

print(f'CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})')
print(f'Individual folds: {cv_scores}')

In [ ]:
import os
# 5.4 Feature Importance
importances = model.feature_importances_
feat_df = pd.DataFrame({'feature': X.columns, 'importance': importances})
feat_df = feat_df.sort_values('importance', ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(x='importance', y='feature', data=feat_df, palette='viridis')
plt.title('Top 15 Feature Importances', fontsize=16)
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.savefig(os.path.join('viz', 'feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()

## 5.5 Evaluation Visualizations
Generate and save all evaluation charts to the `viz/` folder.

In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import seaborn as sns
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize
from itertools import cycle

os.makedirs('viz', exist_ok=True)

# ─── 1. Confusion Matrix ─────────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(cm_norm, cmap='Blues')
plt.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
ax.set_xticks(range(len(le_career.classes_)))
ax.set_yticks(range(len(le_career.classes_)))
ax.set_xticklabels(le_career.classes_, rotation=30, ha='right', fontsize=10)
ax.set_yticklabels(le_career.classes_, fontsize=10)
ax.set_xlabel('Predicted', fontsize=12, labelpad=10)
ax.set_ylabel('Actual', fontsize=12, labelpad=10)
ax.set_title('Confusion Matrix (Normalised)', fontsize=15, fontweight='bold', pad=18)
for i in range(len(le_career.classes_)):
    for j in range(len(le_career.classes_)):
        ax.text(j, i, f'{cm[i,j]}\n({cm_norm[i,j]:.0%})',
                ha='center', va='center', fontsize=9,
                color='white' if cm_norm[i,j] > 0.5 else 'black')
plt.tight_layout()
plt.savefig(os.path.join('viz', 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: viz/confusion_matrix.png')

# ─── 2. Per-Class Precision / Recall / F1 ────────────────────────────────────
p, r, f, s = precision_recall_fscore_support(y_test, y_pred, zero_division=0)
classes = le_career.classes_
x = np.arange(len(classes))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
b1 = ax.bar(x - width, p, width, label='Precision', color='#3b82f6', alpha=0.9, edgecolor='white', linewidth=0.5)
b2 = ax.bar(x,         r, width, label='Recall',    color='#10b981', alpha=0.9, edgecolor='white', linewidth=0.5)
b3 = ax.bar(x + width, f, width, label='F1-Score',  color='#f59e0b', alpha=0.9, edgecolor='white', linewidth=0.5)
for bars in [b1, b2, b3]:
    for bar in bars:
        h = bar.get_height()
        ax.annotate(f'{h:.2f}', xy=(bar.get_x()+bar.get_width()/2, h),
                    xytext=(0,3), textcoords='offset points',
                    ha='center', va='bottom', fontsize=8, color='white')
ax.set_xticks(x)
ax.set_xticklabels(classes, rotation=25, ha='right', fontsize=11)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Per-Class Precision, Recall & F1-Score', fontsize=15, fontweight='bold', pad=18)
ax.legend(fontsize=11)
ax.set_facecolor('#0f172a')
fig.patch.set_facecolor('#1e293b')
ax.tick_params(colors='white')
ax.yaxis.label.set_color('white')
ax.xaxis.label.set_color('white')
ax.title.set_color('white')
ax.legend(fontsize=11, facecolor='#1e293b', edgecolor='none', labelcolor='white')
for spine in ax.spines.values(): spine.set_edgecolor('#334155')
ax.yaxis.grid(True, color='#334155', alpha=0.4)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(os.path.join('viz', 'per_class_metrics.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: viz/per_class_metrics.png')

# ─── 3. Support (class sample counts) ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#E24B4A' if ss < 20 else '#3b82f6' if ss > 200 else '#94a3b8' for ss in s]
bars = ax.barh(classes, s, color=colors, edgecolor='white', linewidth=0.5)
for bar, count in zip(bars, s):
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
            str(count), va='center', fontsize=11, color='white')
ax.set_xlabel('Number of Test Samples', fontsize=12, color='white')
ax.set_title('Class Support (Test Set)', fontsize=15, fontweight='bold', pad=18, color='white')
ax.set_facecolor('#0f172a')
fig.patch.set_facecolor('#1e293b')
ax.tick_params(colors='white')
for spine in ax.spines.values(): spine.set_edgecolor('#334155')
ax.xaxis.grid(True, color='#334155', alpha=0.4)
ax.set_axisbelow(True)
legend_patches = [mpatches.Patch(color='#E24B4A', label='Low support (<20)'),
                  mpatches.Patch(color='#94a3b8', label='Medium support'),
                  mpatches.Patch(color='#3b82f6', label='High support (>200)')]
ax.legend(handles=legend_patches, fontsize=10, facecolor='#1e293b', edgecolor='none', labelcolor='white')
plt.tight_layout()
plt.savefig(os.path.join('viz', 'class_support.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: viz/class_support.png')

# ─── 4. Cross-Validation Bar Chart ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
fold_labels = [f'Fold {i+1}' for i in range(len(cv_scores))]
bar_colors = ['#a78bfa' if v == cv_scores.max() else '#3b82f6' for v in cv_scores]
bars = ax.bar(fold_labels, cv_scores * 100, color=bar_colors, edgecolor='white', linewidth=0.5, width=0.5)
for bar, val in zip(bars, cv_scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val*100:.2f}%', ha='center', fontsize=11, color='white')
ax.axhline(cv_scores.mean() * 100, color='#f59e0b', linewidth=2, linestyle='--',
           label=f'Mean: {cv_scores.mean()*100:.2f}%')
ax.set_ylim(0, 105)
ax.set_ylabel('Accuracy (%)', fontsize=12, color='white')
ax.set_title('5-Fold Cross-Validation Accuracy', fontsize=15, fontweight='bold', pad=18, color='white')
ax.set_facecolor('#0f172a')
fig.patch.set_facecolor('#1e293b')
ax.tick_params(colors='white')
for spine in ax.spines.values(): spine.set_edgecolor('#334155')
ax.yaxis.grid(True, color='#334155', alpha=0.4)
ax.set_axisbelow(True)
ax.legend(fontsize=11, facecolor='#1e293b', edgecolor='none', labelcolor='white')
plt.tight_layout()
plt.savefig(os.path.join('viz', 'cross_validation.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: viz/cross_validation.png')

# ─── 5. Overall Metrics Bar ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
metric_names  = ['Accuracy', 'Precision\n(weighted)', 'Recall\n(weighted)', 'F1-Score\n(weighted)']
metric_values = [accuracy * 100, precision * 100, recall * 100, f1 * 100]
bar_colors    = ['#3b82f6', '#a78bfa', '#10b981', '#f59e0b']
bars = ax.bar(metric_names, metric_values, color=bar_colors, edgecolor='white', linewidth=0.5, width=0.5)
for bar, val in zip(bars, metric_values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.2f}%', ha='center', fontsize=12, fontweight='bold', color='white')
ax.axhline(80, color='#34d399', linewidth=1.5, linestyle=':', alpha=0.7, label='80% baseline')
ax.set_ylim(0, 105)
ax.set_ylabel('Score (%)', fontsize=12, color='white')
ax.set_title('Overall Model Evaluation Metrics', fontsize=15, fontweight='bold', pad=18, color='white')
ax.set_facecolor('#0f172a')
fig.patch.set_facecolor('#1e293b')
ax.tick_params(colors='white')
for spine in ax.spines.values(): spine.set_edgecolor('#334155')
ax.yaxis.grid(True, color='#334155', alpha=0.4)
ax.set_axisbelow(True)
ax.legend(fontsize=11, facecolor='#1e293b', edgecolor='none', labelcolor='white')
plt.tight_layout()
plt.savefig(os.path.join('viz', 'overall_metrics.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: viz/overall_metrics.png')

# ─── 6. Feature Importance ────────────────────────────────────────────────────
importances = model.feature_importances_
feat_df = pd.DataFrame({'feature': X.columns, 'importance': importances})
feat_df = feat_df.sort_values('importance', ascending=True).tail(15)
academic_feats = {'gpa', 'internships', 'projects', 'leadership', 'major_encoded'}
bar_cols = ['#f59e0b' if f in academic_feats else '#3b82f6' for f in feat_df['feature']]

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(feat_df['feature'], feat_df['importance'] * 100,
               color=bar_cols, edgecolor='white', linewidth=0.4)
for bar in bars:
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():.2f}%', va='center', fontsize=9, color='white')
legend_patches = [mpatches.Patch(color='#f59e0b', label='Academic features'),
                  mpatches.Patch(color='#3b82f6', label='Skill features')]
ax.legend(handles=legend_patches, fontsize=10, facecolor='#1e293b', edgecolor='none', labelcolor='white')
ax.set_xlabel('Importance (%)', fontsize=12, color='white')
ax.set_title('Top 15 Feature Importances', fontsize=15, fontweight='bold', pad=18, color='white')
ax.set_facecolor('#0f172a')
fig.patch.set_facecolor('#1e293b')
ax.tick_params(colors='white')
for spine in ax.spines.values(): spine.set_edgecolor('#334155')
ax.xaxis.grid(True, color='#334155', alpha=0.4)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(os.path.join('viz', 'feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: viz/feature_importance.png')

print('\nAll 6 evaluation visualizations saved to viz/')

## 6. Save Model Artifacts

In [ ]:
import os
os.makedirs('models', exist_ok=True)
os.makedirs('viz', exist_ok=True)

# Save model and encoders
joblib.dump(model, os.path.join('models', 'career_predictor_model.pkl'))
joblib.dump(le_career, os.path.join('models', 'label_encoder_career.pkl'))
joblib.dump(le_major, os.path.join('models', 'label_encoder_major.pkl'))
joblib.dump(skills_df.columns.tolist(), os.path.join('models', 'skills_columns.pkl'))

feature_names = X.columns.tolist()
with open(os.path.join('models', 'feature_names.json'), 'w') as f:
    json.dump(feature_names, f)

# Save metrics
metrics = {
    'accuracy': float(accuracy),
    'precision': float(precision),
    'recall': float(recall),
    'f1_score': float(f1),
    'cv_mean': float(cv_scores.mean()),
    'cv_std': float(cv_scores.std()),
    'total_samples': int(len(df)),
    'training_samples': int(X_train.shape[0]),
    'test_samples': int(X_test.shape[0])
}
with open(os.path.join('models', 'model_metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)

# Seed prediction_history.json with sample records
from datetime import datetime, timedelta
import random

sample_users = [
    ('Amaka Okafor', 'Computer Science'),
    ('Chidi Nwachukwu', 'Mathematics'),
    ('Funmi Adeyemi', 'Business'),
    ('Bola Adekunle', 'Engineering'),
    ('Ngozi Eze', 'Psychology'),
    ('Emeka Obiora', 'Biology'),
]
careers = le_career.classes_.tolist()
history = []
base_time = datetime.now()
for i, (name, major) in enumerate(sample_users):
    ts = base_time - timedelta(hours=i*3 + random.randint(0,2))
    history.append({
        'id': i+1,
        'user': name,
        'major': major,
        'career': random.choice(careers),
        'confidence': round(random.uniform(78, 97), 1),
        'timestamp': ts.strftime('%Y-%m-%d %H:%M')
    })

with open(os.path.join('models', 'prediction_history.json'), 'w') as f:
    json.dump(history, f, indent=2)

print('All artifacts saved:')
artifacts = [
    os.path.join('models', x) for x in
    ['career_predictor_model.pkl','label_encoder_career.pkl','label_encoder_major.pkl',
     'skills_columns.pkl','feature_names.json','model_metrics.json','prediction_history.json']
]
for fp in artifacts:
    size = os.path.getsize(fp) if os.path.exists(fp) else 0
    print(f'  {fp} ({size} bytes)')